In [1]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request

In [3]:
import os, time, traceback, urllib.parse, urllib.request
from contextlib import contextmanager
from urllib.error import HTTPError, URLError

def notify_cell(message: str) -> bool:
    """Send a simple Telegram message."""
    token = os.getenv("TELEGRAM_BOT_TOKEN")
    chat_id = os.getenv("TELEGRAM_CHAT_ID")
    if not token or not chat_id:
        print("❌ Missing TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID")
        return False

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    data = urllib.parse.urlencode({"chat_id": chat_id, "text": message}).encode()

    try:
        with urllib.request.urlopen(urllib.request.Request(url, data=data), timeout=20) as r:
            print(f"📨 Sent: {message}")
            return True
    except (HTTPError, URLError) as e:
        print("🚫 Telegram error:", e)
        return False


@contextmanager
def notify_wrap(task_name: str):
    """Context manager to auto-notify on success or failure."""
    start = time.time()
    try:
        yield
    except Exception as e:
        elapsed = time.time() - start
        err_text = ''.join(traceback.format_exception_only(type(e), e)).strip()
        notify_cell(
            f"❌ FAILED: {task_name}\n"
            f"Error: `{err_text}`\n"
            f"Duration: {elapsed/60:.1f} min"
        )
        raise
    else:
        elapsed = time.time() - start
        notify_cell(
            f"✅ DONE: {task_name}\n"
            f"Duration: {elapsed/60:.1f} min"
        )



In [4]:
df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump1.parquet')
df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump2.parquet')
df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump3.parquet')
df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump4.parquet')
df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump5.parquet')
df6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump6.parquet')
df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump7.parquet')

dump1 = df1.to_pandas()
dump2 = df2.to_pandas()
dump3 = df3.to_pandas()
dump4 = df4.to_pandas()
dump5 = df5.to_pandas()
dump6 = df6.to_pandas()
dump7 = df7.to_pandas()

# Load attack dumps
#attack_044h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-044h.parquet").to_pandas()
#attack_080h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-080h.parquet").to_pandas()
#attack_383h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-383h.parquet").to_pandas()
#attack_593h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-593h.parquet").to_pandas()"""


In [5]:
with notify_wrap("Cell 1: hex to bytes and hamming distance computation"):
    def hex_to_bytes(h):
        """Convert hex/bytes/string to bytes."""
        if h is None or (isinstance(h, float) and np.isnan(h)): 
            return b""
        if isinstance(h, (bytes, bytearray)): 
            return bytes(h)
        s = str(h).strip().replace("0x","").replace(" ","")
        if s == "": 
            return b""
        if len(s) % 2 == 1: 
            s = "0"+s
        try: 
            return bytes.fromhex(s)
        except Exception: 
            return str(h).encode("utf-8", errors="ignore")


    def hamming_distance(a: bytes, b: bytes) -> tuple:
        """
        Compute Hamming distance between two byte sequences.
        
        Key difference from Levenshtein:
        - Requires equal length (pads shorter with zeros)
        - Counts number of POSITIONS where bytes differ
        - Much faster: O(n) vs O(m*n)
        
        Args:
            a, b: Byte sequences to compare
            
        Returns:
            Number of differing bytes (0 to max_len)
        """
        len_mismatch = (len(a) != len(b))
    
        # Pad to same length
        max_len = max(len(a), len(b))
        #a_padded = a + b'\x00' * (max_len - len(a))
        #b_padded = b + b'\x00' * (max_len - len(b))
        
        # Count differences
        #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
        
        distance = 0
        for byte_a, byte_b in zip(a, b):
            distance += bin(byte_a ^ byte_b).count('1')
        return (distance, len_mismatch)
        

📨 Sent: ✅ DONE: Cell 1: hex to bytes and hamming distance computation
Duration: 0.0 min


In [6]:
with notify_wrap("Cell 1: compute Hamming distances of dumps"):
    def compute_hamming_distances(dumps, out_csv, process_per_dump=True):
        """
        Compute Hamming distances between consecutive payloads.
        
        Args:
            dumps: List of tuples (dump_name, dataframe)
                DataFrame should have: timestamp, arbitration_id, data, [label]
            out_csv: Output CSV path
            process_per_dump: If True, reset history between dumps
        
        Returns:
            DataFrame with columns: dump, timestamp, arbitration_id, 
                                ham_dist, ham_norm, label
        """
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        
        records = []
        prev_payload = {}  # {arbitration_id: previous_bytes}
        
        for dump_name, df in dumps:
            d = df.copy()
            
            # Ensure timestamp column
            if "timestamp" not in d.columns:
                if d.index.name == "timestamp":
                    d = d.reset_index()
                else:
                    d = d.reset_index().rename(columns={"index": "timestamp"})
            
            d = d.sort_values("timestamp", kind="mergesort")
            has_label = "label" in d.columns
            
            for _, row in d.iterrows():
                aid = row["arbitration_id"]
                ts = row["timestamp"]
                lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
                curr_bytes = hex_to_bytes(row["data"])
                
                # Get previous payload for this ID
                prev = prev_payload.get(aid)
                
                if prev is not None:
                    # Compute Hamming distance
                    dist, len_mismatch = hamming_distance(curr_bytes, prev)
                    #max_len = max(len(curr_bytes), len(prev))
                    #norm_dist = dist / (max_len*8) if max_len > 0 else 0.0
                    
                    records.append({
                        "dump": dump_name,
                        "timestamp": ts,
                        "arbitration_id": aid,
                        "payload_len": len(curr_bytes),
                        "ham_dist": dist,
                        #"ham_norm": norm_dist,
                        "len_mismatch": len_mismatch,  
                        "label": lab
                    })
                
                # Update previous payload
                prev_payload[aid] = curr_bytes
            
            if process_per_dump:
                prev_payload.clear()
        
        out_df = pd.DataFrame.from_records(records)
        out_df.to_csv(out_csv, index=False)
        print(f"✅ Saved: {out_csv} (rows={len(out_df):,})")
        
        return out_df

📨 Sent: ✅ DONE: Cell 1: compute Hamming distances of dumps
Duration: 0.0 min


In [7]:
def build_hamming_range_model(hamming_csv, output_csv):
    
    #Learn [min, max] Hamming distance range per ID from benign data.
    
    df = pd.read_csv(hamming_csv)
    
    # Group by ID and compute min/max
    ranges = (df.groupby('arbitration_id')['ham_dist']
            .agg(['min', 'max', 'count'])
            .reset_index())
    
    ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
    ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
    
    # Normalize (max 64 bits for 8-byte payload)
    ranges['min_norm'] = ranges['min_hamming'] / 64.0
    ranges['max_norm'] = ranges['max_hamming'] / 64.0
    ranges['range_norm'] = ranges['range_size'] / 64.0
    
    # Classify (σ=6 bits from paper)
    sigma = 6
    def classify(r):
        if r == 0:
            return 'NoRange'
        elif r <= sigma:
            return 'SmallRange'
        else:
            return 'MidRange'
    
    ranges['class'] = ranges['range_size'].apply(classify)
    
    # Save
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    ranges.to_csv(output_csv, index=False)
    
    # Print summary
    print(f"\n Model built: {len(ranges)} IDs")
    print(f"   NoRange: {(ranges['class']=='NoRange').sum()}")
    print(f"   SmallRange: {(ranges['class']=='SmallRange').sum()}")
    print(f"   MidRange: {(ranges['class']=='MidRange').sum()}")
    print(f"   Saved: {output_csv}")
    
    return ranges

In [8]:
"""with notify_wrap("Cell 3: simple detection with Hamming distances"):
    def detect_simple(test_csv, ranges_csv, output_csv="artifacts/detections_simple.csv"):
        
        Simple detection: Flag if distance is OUTSIDE [min, max] range.
        No windowing - direct per-message detection.
        
        Args:
            test_csv: Path to test data with Hamming distances
            ranges_csv: Path to learned ranges
            output_csv: Where to save results
            
        Returns:
            DataFrame with detection results and performance metrics
        
        print("="*70)
        print("🚨 Simple Detection (1 Anomaly = 1 Alarm)")
        print("="*70)
        
        # Load data
        test = pd.read_csv(test_csv)
        ranges = pd.read_csv(ranges_csv)
        
        # Merge test data with ranges
        test_with_ranges = test.merge(
            ranges[['arbitration_id', 'min_hamming', 'max_hamming', 'range_size', 'class', 'expected_tpr']],
            on='arbitration_id',
            how='left'
        )
        
        # Detect: Flag if OUTSIDE range [min, max]
        test_with_ranges['is_anomaly'] = (
            (test_with_ranges['ham_dist'] < test_with_ranges['min_hamming']) |
            (test_with_ranges['ham_dist'] > test_with_ranges['max_hamming'])
        )
        
        # Count
        n_anomalies = test_with_ranges['is_anomaly'].sum()
        n_total = len(test_with_ranges)
        
        print(f"\n📊 Detection Summary:")
        print(f"   Total messages:    {n_total:,}")
        print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
        
        # Compute metrics if labels available
        metrics = {}
        if 'label' in test_with_ranges.columns:
            y_true = test_with_ranges['label'].values
            y_pred = test_with_ranges['is_anomaly'].values
            
            TP = ((y_true == 1) & (y_pred == True)).sum()
            FP = ((y_true == 0) & (y_pred == True)).sum()
            TN = ((y_true == 0) & (y_pred == False)).sum()
            FN = ((y_true == 1) & (y_pred == False)).sum()
            
            tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
            fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
            
            print(f"\n📊 Confusion Matrix:")
            print(f"   TP (attacks detected):  {TP:,}")
            print(f"   FP (false alarms):      {FP:,}")
            print(f"   TN (benign correct):    {TN:,}")
            print(f"   FN (attacks missed):    {FN:,}")
            
            print(f"\n📈 Performance Metrics:")
            print(f"   TPR (Detection Rate): {tpr*100:.2f}%")
            print(f"   FPR (False Alarms):   {fpr*100:.2f}%")
            print(f"   Precision:            {precision*100:.2f}%")
            print(f"   F1-Score:             {f1*100:.2f}%")
            
            metrics['overall'] = {
                'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
                'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
            }
            
            # Per-class breakdown (paper's key result)
            print(f"\n📋 Detection Rate by ID Class (Paper's Expected vs Actual):")
            print(f"   {'Class':<12} {'Expected':<12} {'Actual':<12} {'IDs':<8}")
            print(f"   {'-'*50}")
            
            for cls in ['NoRange', 'SmallRange', 'MidRange']:
                cls_data = test_with_ranges[test_with_ranges['class'] == cls]
                if len(cls_data) > 0 and 'label' in cls_data.columns:
                    cls_attacks = cls_data[cls_data['label'] == 1]
                    cls_detected = cls_attacks[cls_attacks['is_anomaly']]
                    
                    if len(cls_attacks) > 0:
                        actual_tpr = len(cls_detected) / len(cls_attacks)
                        expected_tpr = cls_data['expected_tpr'].iloc[0]
                        n_ids = cls_data['arbitration_id'].nunique()
                        
                        print(f"   {cls:12s} {expected_tpr*100:>6.1f}%      {actual_tpr*100:>6.1f}%      {n_ids:>3d}")
            
            # Per-ID breakdown
            per_id = []
            for aid in test_with_ranges['arbitration_id'].unique():
                aid_data = test_with_ranges[test_with_ranges['arbitration_id'] == aid]
                
                if 'label' not in aid_data.columns:
                    continue
                
                y_t = aid_data['label'].values
                y_p = aid_data['is_anomaly'].values
                
                tp = ((y_t == 1) & (y_p == True)).sum()
                fn = ((y_t == 1) & (y_p == False)).sum()
                tpr_id = tp / (tp + fn) if (tp + fn) > 0 else 0
                
                aid_class = aid_data['class'].iloc[0] if len(aid_data) > 0 else 'Unknown'
                aid_range = aid_data['range_size'].iloc[0] if len(aid_data) > 0 else np.nan
                
                per_id.append({
                    'arbitration_id': aid,
                    'aid_hex': f"0x{aid:03X}",
                    'n_attacks': int(tp + fn),
                    'detected': int(tp),
                    'missed': int(fn),
                    'tpr': tpr_id,
                    'class': aid_class,
                    'range_size': aid_range
                })
            
            metrics['per_id'] = pd.DataFrame(per_id)
        
        # Save results
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        test_with_ranges.to_csv(output_csv, index=False)
        print(f"\n✅ Results saved to: {output_csv}")
        print("="*70)
        
        return test_with_ranges, metrics
"""

'with notify_wrap("Cell 3: simple detection with Hamming distances"):\n    def detect_simple(test_csv, ranges_csv, output_csv="artifacts/detections_simple.csv"):\n\n        Simple detection: Flag if distance is OUTSIDE [min, max] range.\n        No windowing - direct per-message detection.\n\n        Args:\n            test_csv: Path to test data with Hamming distances\n            ranges_csv: Path to learned ranges\n            output_csv: Where to save results\n\n        Returns:\n            DataFrame with detection results and performance metrics\n\n        print("="*70)\n        print("🚨 Simple Detection (1 Anomaly = 1 Alarm)")\n        print("="*70)\n\n        # Load data\n        test = pd.read_csv(test_csv)\n        ranges = pd.read_csv(ranges_csv)\n\n        # Merge test data with ranges\n        test_with_ranges = test.merge(\n            ranges[[\'arbitration_id\', \'min_hamming\', \'max_hamming\', \'range_size\', \'class\', \'expected_tpr\']],\n            on=\'arbitration_

In [9]:
def detect_simple(test_csv, ranges_csv, output_csv):
    """
    Detect anomalies: Flag if distance is OUTSIDE [min, max] range.
    """
    # Load data
    test = pd.read_csv(test_csv)
    ranges = pd.read_csv(ranges_csv)
    
    # Merge test data with ranges
    test_with_ranges = test.merge(
        ranges[['arbitration_id', 'min_hamming', 'max_hamming', 'range_size', 'class']],
        on='arbitration_id',
        how='left'
    )
    
    # Detect: Flag if OUTSIDE range [min, max]
    test_with_ranges['is_anomaly'] = (
        test_with_ranges['len_mismatch'] |
        (test_with_ranges['ham_dist'] < test_with_ranges['min_hamming']) |
        (test_with_ranges['ham_dist'] > test_with_ranges['max_hamming'])
    )
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in test_with_ranges.columns:
        y_true = test_with_ranges['label'].values
        y_pred = test_with_ranges['is_anomaly'].values
        
        TP = ((y_true == 1) & (y_pred == True)).sum()
        FP = ((y_true == 0) & (y_pred == True)).sum()
        TN = ((y_true == 0) & (y_pred == False)).sum()
        FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\n Detection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
    
    # Save results
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    test_with_ranges.to_csv(output_csv, index=False)
    
    return test_with_ranges, metrics

In [10]:
if __name__ == "__main__":
    print("\n" + "="*70)
    print("🚀 HAMMING DISTANCE IDS - REPLAY ATTACKS")
    print("="*70)
    
    files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw"
    
    benign_dumps = [
        ("dump1", dump1),
        ("dump2", dump2),
        ("dump3", dump3),
        ("dump4", dump4),
        ("dump5", dump5),
        ("dump6", dump6),
        ("dump7", dump7)
    ]
    
    benign_ham = compute_hamming_distances(
        benign_dumps, 
        "artifacts/benign_hamming.csv",
        process_per_dump=False
    )
    
    print("\n" + "="*70)
    print("HAMMING DISTANCE COMPUTED FOR BENIGN DUMPS")
    print("="*70)
    
    
    replay_files = [f for f in os.listdir(files_pathname) 
                    if f.startswith("dump6-repl-") and f.endswith(".parquet")]
    
    # Extract the range part (e.g., "dump6-repl-0-120.parquet" -> "0-120")
    replay_ids = sorted([f.replace("dump6-repl-", "").replace(".parquet", "") 
                        for f in replay_files])
    
    print(f"\n Found {len(replay_ids)} replay attack files:")
    print(f"   {replay_ids}")
    
    ranges = build_hamming_range_model(
    "artifacts/benign_hamming.csv",
    "artifacts/hamming_ranges_rep.csv"
)
    results_summary = []
    
    for i, replay_id in enumerate(replay_ids, 1):
        print(f"\n[{i}/{len(replay_ids)}] Testing: dump6-repl-{replay_id}")
        
        try:
            # Load attack file
            attack_file = os.path.join(files_pathname, f"dump6-repl-{replay_id}.parquet")
            attack_df = pq.read_table(attack_file).to_pandas()
            
            print(f"   Loaded: {len(attack_df):,} messages")
            
            # Compute Hamming
            attack_dumps = [(f"repl_{replay_id}", attack_df)]
            attack_ham = compute_hamming_distances(
                attack_dumps,
                f"artifacts/attack_ham_repl_{replay_id}.csv",
                process_per_dump=True
            )
            
            # Detect
            results, metrics = detect_simple(
                f"artifacts/attack_ham_repl_{replay_id}.csv",
                "artifacts/hamming_ranges_rep.csv",
                f"artifacts/detections_repl_{replay_id}.csv"
            )
            
            # Store results
            if 'overall' in metrics:
                results_summary.append({
                    'replay_id': replay_id,
                    'n_messages': len(attack_df),
                    'n_attacks': metrics['overall']['TP'] + metrics['overall']['FN'],
                    'TPR': metrics['overall']['TPR'] * 100,
                    'FPR': metrics['overall']['FPR'] * 100,
                    'Precision': metrics['overall']['Precision'] * 100,
                    'F1': metrics['overall']['F1'] * 100,
                    'TP': metrics['overall']['TP'],
                    'FP': metrics['overall']['FP'],
                    'TN': metrics['overall']['TN'],
                    'FN': metrics['overall']['FN']
                })
                
                print(f"   ✅ TPR: {metrics['overall']['TPR']*100:.1f}%, "
                      f"FPR: {metrics['overall']['FPR']*100:.2f}%, "
                      f"F1: {metrics['overall']['F1']*100:.1f}%")
        
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ========================================================================
    # STEP 4: Summary and Comparison
    # ========================================================================
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv("artifacts/all_replay_summary.csv", index=False)
    
    print("\n" + "="*70)
    print("REPLAY ATTACKS - FINAL SUMMARY")
    print("="*70)
    print(f"\nTested: {len(summary_df)}/{len(replay_ids)} attack types")
    
    if len(summary_df) > 0:
        print(f"\nOverall Performance:")
        print(f"   Mean TPR:       {summary_df['TPR'].mean():.2f}%")
        print(f"   Median TPR:     {summary_df['TPR'].median():.2f}%")
        print(f"   Min TPR:        {summary_df['TPR'].min():.2f}%")
        print(f"   Max TPR:        {summary_df['TPR'].max():.2f}%")
        print(f"   Mean FPR:       {summary_df['FPR'].mean():.4f}%")
        print(f"   Mean F1:        {summary_df['F1'].mean():.2f}%")
        
        print("\nResults by Replay Range:")
        print(summary_df[['replay_id', 'n_attacks', 'TPR', 'FPR', 'F1']].to_string(index=False))
        





🚀 HAMMING DISTANCE IDS - REPLAY ATTACKS
✅ Saved: artifacts/benign_hamming.csv (rows=27,266,784)

HAMMING DISTANCE COMPUTED FOR BENIGN DUMPS

 Found 4 replay attack files:
   ['0-120', '120-240', '240-360', '360-479.99999']

 Model built: 64 IDs
   NoRange: 11
   SmallRange: 14
   MidRange: 39
   Saved: artifacts/hamming_ranges_rep.csv

[1/4] Testing: dump6-repl-0-120
   Loaded: 6,169,282 messages
✅ Saved: artifacts/attack_ham_repl_0-120.csv (rows=6,169,218)

 Detection Results:
   TPR: 1.63% | FPR: 0.70% | F1: 3.15%
   TP: 30,713 | FP: 29,902 | TN: 4,249,943 | FN: 1,858,660
   ✅ TPR: 1.6%, FPR: 0.70%, F1: 3.2%

[2/4] Testing: dump6-repl-120-240
   Loaded: 6,179,536 messages
✅ Saved: artifacts/attack_ham_repl_120-240.csv (rows=6,179,472)

 Detection Results:
   TPR: 1.64% | FPR: 0.73% | F1: 3.17%
   TP: 31,096 | FP: 31,176 | TN: 4,248,669 | FN: 1,868,531
   ✅ TPR: 1.6%, FPR: 0.73%, F1: 3.2%

[3/4] Testing: dump6-repl-240-360
   Loaded: 6,178,108 messages
✅ Saved: artifacts/attack_ham_r

In [11]:
notify_cell("🚀 Hamming Distance IDS Pipeline Completed Successfully!")

📨 Sent: 🚀 Hamming Distance IDS Pipeline Completed Successfully!


True